# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [131]:
import importlib.util
from pathlib import Path

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [132]:
# Parâmetros importados do arquivo config.py
print(
    "Filtrar por quantidade de alelos TCC2:.........................",
    parametros.filtarar_por_qte_de_alelos_tcc2,
)
print(
    "Parâmetro de filtragem median binding percentile TCC2:.........",
    parametros.parametro_de_filtragem_mbp_tcc2,
)
print(
    "Percentual de match mínimo TCC2:...............................",
    parametros.percent_match_minimo_tcc2,
)

Filtrar por quantidade de alelos TCC2:......................... 2
Parâmetro de filtragem median binding percentile TCC2:......... 1
Percentual de match mínimo TCC2:............................... 95.0


In [133]:
import pandas as pd

In [134]:
df = pd.read_csv('./T CELL/DENV 1 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el-4.3 core,netmhciipan_el-4.3 score,netmhciipan_el-4.3 percentile
0,1,KNETWKLARASFIE,206,219,14,HLA-DRB1*09:01,110,0.07,WKLARASFI,0.911329,0.07
1,1,QYKFQADSPKRLSA,31,44,14,HLA-DRB3*01:01,75,0.08,FQADSPKRL,0.920957,0.08
2,1,KNETWKLARASFIEVK,206,221,16,HLA-DRB1*09:01,246,0.08,WKLARASFI,0.910584,0.08
3,1,QYKFQADSPKRLS,31,43,13,HLA-DRB3*01:01,7,0.09,FQADSPKRL,0.915925,0.09
4,1,QYKFQADSPKRLSAA,31,45,15,HLA-DRB3*01:01,143,0.10,FQADSPKRL,0.912478,0.10
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,WCCRSCTLPPLRFKGEDGCW,311,330,20,HLA-DRB1*04:01,537,100.00,CRSCTLPPL,0.000000,100.00
16412,1,WCCRSCTLPPLRFKGEDGCW,311,330,20,HLA-DRB1*04:05,537,100.00,PLRFKGEDG,0.000000,100.00
16413,1,WCCRSCTLPPLRFKGEDGCW,311,330,20,HLA-DRB3*01:01,537,100.00,LRFKGEDGC,0.000000,100.00
16414,1,WCCRSCTLPPLRFKGEDGCWY,311,331,21,HLA-DRB1*04:01,604,100.00,FKGEDGCWY,0.000000,100.00


## Selecionando Epítopos por median binding percentile.

In [135]:
mbp_minimo = parametros.parametro_de_filtragem_mbp_tcc2

In [136]:
df_mbp_m5 = df[df['median binding percentile'] < mbp_minimo].copy()
print("Filtrando por median binding percentile < ", mbp_minimo)
df_mbp_m5

Filtrando por median binding percentile <  1


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el-4.3 core,netmhciipan_el-4.3 score,netmhciipan_el-4.3 percentile
0,1,KNETWKLARASFIE,206,219,14,HLA-DRB1*09:01,110,0.07,WKLARASFI,0.911329,0.07
1,1,QYKFQADSPKRLSA,31,44,14,HLA-DRB3*01:01,75,0.08,FQADSPKRL,0.920957,0.08
2,1,KNETWKLARASFIEVK,206,221,16,HLA-DRB1*09:01,246,0.08,WKLARASFI,0.910584,0.08
3,1,QYKFQADSPKRLS,31,43,13,HLA-DRB3*01:01,7,0.09,FQADSPKRL,0.915925,0.09
4,1,QYKFQADSPKRLSAA,31,45,15,HLA-DRB3*01:01,143,0.10,FQADSPKRL,0.912478,0.10
...,...,...,...,...,...,...,...,...,...,...,...
111,1,ADVQNTTFIIDGP,126,138,13,HLA-DPA1*02:01/DPB1*01:01,26,0.98,VQNTTFIID,0.608050,0.98
112,1,ENDMKFTVVVGDVSGILA,81,98,18,HLA-DPA1*02:01/DPB1*14:01,357,0.98,KFTVVVGDV,0.592744,0.98
113,1,HTWTEQYKFQADSPKRLSAA,26,45,20,HLA-DRB5*01:01,480,0.98,YKFQADSPK,0.423307,0.98
114,1,HTWTEQYKFQADSPKR,26,41,16,HLA-DRB5*01:01,210,0.99,YKFQADSPK,0.631851,0.99


## Agrupando por pepitideos e agregando colunas pertinentes

In [137]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ADMGYWIESEKNET,196,209,1,0.670,HLA-DPA1*02:01/DPB1*05:01
1,ADMGYWIESEKNETW,196,210,1,0.850,HLA-DPA1*02:01/DPB1*05:01
2,ADVQNTTFIIDGP,126,138,2,0.710,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
3,ADVQNTTFIIDGPN,126,139,1,0.420,HLA-DPA1*01:03/DPB1*04:01
4,ADVQNTTFIIDGPNT,126,140,1,0.640,HLA-DPA1*01:03/DPB1*04:01
...,...,...,...,...,...,...
74,QYKFQADSPKRLSAAIGKAW,31,50,1,0.550,HLA-DRB3*01:01
75,QYKFQADSPKRLSAAIGKAWE,31,51,1,0.630,HLA-DRB3*01:01
76,TRLENIMWKQISN,61,73,2,0.765,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
77,TRLENIMWKQISNE,61,74,1,0.820,HLA-DPA1*01:03/DPB1*02:01


## Filtragem por qte_de_alelos

In [138]:
# mbp_minimo = 1
qte_alelos_minimo = parametros.filtarar_por_qte_de_alelos_tcc2

In [139]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= qte_alelos_minimo
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ADVQNTTFIIDGP,126,138,2,0.710,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
1,AKIIGADVQNTTF,121,133,2,0.375,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
2,AKIIGADVQNTTFI,121,134,2,0.735,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
3,GIFTTNIWLKLRD,161,173,3,0.500,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
4,GIFTTNIWLKLRDS,161,174,3,0.650,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
5,HKYSWKSWGKAKIIG,111,125,2,0.740,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
6,HTWTEQYKFQADSPKRL,26,42,2,0.825,"HLA-DRB3*01:01, HLA-DRB5*01:01"
7,HTWTEQYKFQADSPKRLS,26,43,2,0.430,"HLA-DRB3*01:01, HLA-DRB5*01:01"
8,HTWTEQYKFQADSPKRLSA,26,44,2,0.500,"HLA-DRB3*01:01, HLA-DRB5*01:01"
9,HTWTEQYKFQADSPKRLSAA,26,45,2,0.575,"HLA-DRB3*01:01, HLA-DRB5*01:01"


## Sorting por median_biding_percentile

In [140]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,KNETWKLARASFIEVK,206,221,3,0.220,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
1,KNETWKLARASFIE,206,219,3,0.260,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
2,KNETWKLARASFIEV,206,220,3,0.260,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
3,PECPDNQRAWNIWEVEDYG,141,159,2,0.345,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
4,KNETWKLARASFIEVKT,206,222,3,0.360,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
5,PECPDNQRAWNIWEVEDYGF,141,160,2,0.370,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
6,AKIIGADVQNTTF,121,133,2,0.375,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
7,KNETWKLARASFI,206,218,2,0.375,"HLA-DRB1*07:01, HLA-DRB1*09:01"
8,PECPDNQRAWNIWEVEDYGFG,141,161,2,0.375,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
9,HTWTEQYKFQADSPKRLS,26,43,2,0.430,"HLA-DRB3*01:01, HLA-DRB5*01:01"


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [141]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_2.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0          KNETWKLARASFIEVK
1            KNETWKLARASFIE
2           KNETWKLARASFIEV
3       PECPDNQRAWNIWEVEDYG
4         KNETWKLARASFIEVKT
5      PECPDNQRAWNIWEVEDYGF
6             AKIIGADVQNTTF
7             KNETWKLARASFI
8     PECPDNQRAWNIWEVEDYGFG
9        HTWTEQYKFQADSPKRLS
10           QYKFQADSPKRLSA
11          QYKFQADSPKRLSAA
12            QYKFQADSPKRLS
13    WIESEKNETWKLARASFIEVK
14            GIFTTNIWLKLRD
15      HTWTEQYKFQADSPKRLSA
16       KNETWKLARASFIEVKTC
17       PECPDNQRAWNIWEVEDY
18     HTWTEQYKFQADSPKRLSAA
19      KNETWKLARASFIEVKTCI
20            NQRAWNIWEVEDY
21           GIFTTNIWLKLRDS
22        PECPDNQRAWNIWEVED
23            ADVQNTTFIIDGP
24           AKIIGADVQNTTFI
25          HKYSWKSWGKAKIIG
26    PQPMEHKYSWKSWGKAKIIGA
27            TRLENIMWKQISN
28        HTWTEQYKFQADSPKRL
29      KSWGKAKIIGADVQNTTFI
Name: peptide, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [142]:
# !seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [143]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,KNETWKLARASFIEVK,16,89.46% (908/1015),68.75%,100.00%,NaN
1,2,NP 2,KNETWKLARASFIE,14,89.46% (908/1015),64.29%,100.00%,NaN
2,3,NP 3,KNETWKLARASFIEV,15,89.46% (908/1015),66.67%,100.00%,NaN
3,4,NP 4,PECPDNQRAWNIWEVEDYG,19,9.56% (97/1015),78.95%,100.00%,NaN
4,5,NP 5,KNETWKLARASFIEVKT,17,89.46% (908/1015),70.59%,100.00%,NaN
5,6,NP 6,PECPDNQRAWNIWEVEDYGF,20,9.56% (97/1015),80.00%,100.00%,NaN
6,7,NP 7,AKIIGADVQNTTF,13,49.56% (503/1015),53.85%,100.00%,NaN
7,8,NP 8,KNETWKLARASFI,13,89.46% (908/1015),61.54%,100.00%,NaN
8,9,NP 9,PECPDNQRAWNIWEVEDYGFG,21,9.56% (97/1015),80.95%,100.00%,NaN
9,10,NP 10,HTWTEQYKFQADSPKRLS,18,99.01% (1005/1015),94.44%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [144]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# MPB
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "median_binding_percentile"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# start
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "start"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# end
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "end"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos
0,NP 1,KNETWKLARASFIEVK,16,89.46% (908/1015),68.75%,100.00%,3,0.220,206,221,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
1,NP 2,KNETWKLARASFIE,14,89.46% (908/1015),64.29%,100.00%,3,0.260,206,219,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
2,NP 3,KNETWKLARASFIEV,15,89.46% (908/1015),66.67%,100.00%,3,0.260,206,220,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
3,NP 4,PECPDNQRAWNIWEVEDYG,19,9.56% (97/1015),78.95%,100.00%,2,0.345,141,159,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
4,NP 5,KNETWKLARASFIEVKT,17,89.46% (908/1015),70.59%,100.00%,3,0.360,206,222,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
5,NP 6,PECPDNQRAWNIWEVEDYGF,20,9.56% (97/1015),80.00%,100.00%,2,0.370,141,160,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
6,NP 7,AKIIGADVQNTTF,13,49.56% (503/1015),53.85%,100.00%,2,0.375,121,133,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
7,NP 8,KNETWKLARASFI,13,89.46% (908/1015),61.54%,100.00%,2,0.375,206,218,"HLA-DRB1*07:01, HLA-DRB1*09:01"
8,NP 9,PECPDNQRAWNIWEVEDYGFG,21,9.56% (97/1015),80.95%,100.00%,2,0.375,141,161,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
9,NP 10,HTWTEQYKFQADSPKRLS,18,99.01% (1005/1015),94.44%,100.00%,2,0.430,26,43,"HLA-DRB3*01:01, HLA-DRB5*01:01"


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [145]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 1,KNETWKLARASFIEVK,16,89.46% (908/1015),68.75%,100.00%,3,0.220,206,221,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01",89.46
1,NP 2,KNETWKLARASFIE,14,89.46% (908/1015),64.29%,100.00%,3,0.260,206,219,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01",89.46
2,NP 3,KNETWKLARASFIEV,15,89.46% (908/1015),66.67%,100.00%,3,0.260,206,220,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01",89.46
3,NP 4,PECPDNQRAWNIWEVEDYG,19,9.56% (97/1015),78.95%,100.00%,2,0.345,141,159,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1...",9.56
4,NP 5,KNETWKLARASFIEVKT,17,89.46% (908/1015),70.59%,100.00%,3,0.360,206,222,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01",89.46
5,NP 6,PECPDNQRAWNIWEVEDYGF,20,9.56% (97/1015),80.00%,100.00%,2,0.370,141,160,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1...",9.56
6,NP 7,AKIIGADVQNTTF,13,49.56% (503/1015),53.85%,100.00%,2,0.375,121,133,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1...",49.56
7,NP 8,KNETWKLARASFI,13,89.46% (908/1015),61.54%,100.00%,2,0.375,206,218,"HLA-DRB1*07:01, HLA-DRB1*09:01",89.46
8,NP 9,PECPDNQRAWNIWEVEDYGFG,21,9.56% (97/1015),80.95%,100.00%,2,0.375,141,161,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1...",9.56
9,NP 10,HTWTEQYKFQADSPKRLS,18,99.01% (1005/1015),94.44%,100.00%,2,0.430,26,43,"HLA-DRB3*01:01, HLA-DRB5*01:01",99.01


### Sort e filtragem por percent_match

In [146]:
minimum_percent_match = parametros.percent_match_minimo_tcc2

In [147]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= minimum_percent_match]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 29,HTWTEQYKFQADSPKRL,17,99.70% (1012/1015),94.12%,100.00%,2,0.825,26,42,"HLA-DRB3*01:01, HLA-DRB5*01:01",99.70
1,NP 13,QYKFQADSPKRLS,13,99.11% (1006/1015),92.31%,100.00%,2,0.450,31,43,"HLA-DRB3*01:01, HLA-DRB3*02:02",99.11
2,NP 10,HTWTEQYKFQADSPKRLS,18,99.01% (1005/1015),94.44%,100.00%,2,0.430,26,43,"HLA-DRB3*01:01, HLA-DRB5*01:01",99.01
3,NP 11,QYKFQADSPKRLSA,14,98.72% (1002/1015),85.71%,100.00%,2,0.440,31,44,"HLA-DRB3*01:01, HLA-DRB3*02:02",98.72
4,NP 12,QYKFQADSPKRLSAA,15,98.72% (1002/1015),86.67%,100.00%,2,0.440,31,45,"HLA-DRB3*01:01, HLA-DRB3*02:02",98.72
5,NP 16,HTWTEQYKFQADSPKRLSA,19,98.62% (1001/1015),89.47%,100.00%,2,0.500,26,44,"HLA-DRB3*01:01, HLA-DRB5*01:01",98.62
6,NP 19,HTWTEQYKFQADSPKRLSAA,20,98.62% (1001/1015),90.00%,100.00%,2,0.575,26,45,"HLA-DRB3*01:01, HLA-DRB5*01:01",98.62
7,NP 28,TRLENIMWKQISN,13,95.86% (973/1015),69.23%,100.00%,2,0.765,61,73,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",95.86


### Estrutura secundária

In [148]:
def processar_horiz(caminho_horiz):
    """
    Lê o arquivo .horiz do PSIPRED e retorna um dicionário 
    contendo as strings de confiança ('conf'), predição ('pred') e sequência ('seq').
    """
    confianca = ""
    predicao = ""
    sequencia = ""
    
    with open(caminho_horiz, 'r') as f:
        for linha in f:
            linha_str = linha.strip()
            if linha_str.startswith("Conf:"):
                confianca += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("Pred:"):
                predicao += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("AA:"):
                sequencia += linha_str.split(":", 1)[1].strip()
                
    return {
        "conf": confianca,
        "pred": predicao,
        "seq": sequencia
    }

In [149]:
denv_horiz = processar_horiz("./PSIPRED/denv.horiz")
denv_horiz

{'conf': '9013886999939925069997554001021479749988999999999973938999698999999996388999999877985799988722046145131014898520153204401023577789479980899987765453897799999567887505999866444555892562167009978874264488861289479999999643764379412568888631112058565788543446899641358893138189988317999799789889997531346269961362630579999869995994061134671687656437855329',
 'pred': 'CCCCEEECCCCEEEECCEEEEECCHHCCCCEEEEEECCHHHHHHHHHHHHHCCCEEEEECCHHHHHHHHHHHHHHHHHHHHCCCCEEEEECCCCEEEECCCEEECCCCCCCCCEECCCCCCCEECCCCCCEEEEEECCCCCCCCCCCEEEEEEEEEEEEEEEEECEEEEEECCCCCCCCCHHHHHHHEECCEEEEECCCEEEEECCCCCEEEEEEEEEEEEEEECCCCCCCCCCCCCCEECCCCHHHCCCCCCCCCCCCCCCCCCCCCCCCCEEEEEEECCCCEEEECCCCCCCCCCCEEEECCCCCCCCEECCCCCCCCEEEECCCCEECCEEEECCCCCCCCEEEEEECC',
 'seq': 'DSGCVINWKGRELKCGSGIFVTNEVHTWTEQYKFQADSPKRLSAAIGKAWEEGVCGIRSATRLENIMWKQISNELNHILLENDMKFTVVVGDVSGILAQGKKMIRPQPMEHKYSWKSWGKAKIIGADVQNTTFIIDGPNTPECPDNQRAWNIWEVEDYGFGIFTTNIWLKLRDSYTQVCDHRLMSAAIKDSKAVHADMGYWIESEKNETWKLARASFIEVKTCIWPKSHTLWSNGVLESEMIIPKIYGGPISQHNYRPGYF

In [150]:
def avaliar_epitopos_com_horiz(df_epitopos, conf, pred, seq):
    """Cruza o DataFrame de epítopos com as strings de predição do .horiz já processadas."""
    resultados = []
    
    for idx, row in df_epitopos.iterrows():
        # Identifica a coluna correta do peptídeo dependendo do estágio do pipeline
        col_pep = 'peptide' if 'peptide' in df_epitopos.columns else 'Epitope sequence'
        peptideo = row[col_pep]
        start = int(row['start'])
        end = int(row['end'])
        
        # Ajuste para índice em Python (base 0)
        idx_inicio = start - 1
        idx_fim = end
        
        # Extrai os trechos correspondentes nas strings do horiz
        sub_pred = pred[idx_inicio:idx_fim]
        sub_conf = conf[idx_inicio:idx_fim]
        
        # Conta a proporção de cada estrutura secundária
        # 'H' = Hélice, 'E' = Fita Beta, 'C' ou '.' = Coil/Alça
        n_coils = sub_pred.count('C') + sub_pred.count('.')
        n_helix = sub_pred.count('H')
        n_strand = sub_pred.count('E')
        
        resultados.append({
            'Epitope': peptideo,
            'Start': start,
            'End': end,
            'Pred_SS': sub_pred,
            'Coils': n_coils,
            'Helices': n_helix,
            'Strands': n_strand
        })
        
    return pd.DataFrame(resultados)

In [151]:
conf = denv_horiz["conf"]
pred = denv_horiz["pred"]
seq = denv_horiz["seq"]

avaliacao_epitopos = avaliar_epitopos_com_horiz(conservacy_result_filtered, conf, pred, seq)
avaliacao_epitopos

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands
0,HTWTEQYKFQADSPKRL,26,42,HCCCCEEEEEECCHHHH,6,5,6
1,QYKFQADSPKRLS,31,43,EEEEEECCHHHHH,2,5,6
2,HTWTEQYKFQADSPKRLS,26,43,HCCCCEEEEEECCHHHHH,6,6,6
3,QYKFQADSPKRLSA,31,44,EEEEEECCHHHHHH,2,6,6
4,QYKFQADSPKRLSAA,31,45,EEEEEECCHHHHHHH,2,7,6
5,HTWTEQYKFQADSPKRLSA,26,44,HCCCCEEEEEECCHHHHHH,6,7,6
6,HTWTEQYKFQADSPKRLSAA,26,45,HCCCCEEEEEECCHHHHHHH,6,8,6
7,TRLENIMWKQISN,61,73,CHHHHHHHHHHHH,1,12,0


In [152]:
# Calcula o percentual de Coils (flexibilidade) para cada epítopo
avaliacao_epitopos['Percent_Coils'] = (
    avaliacao_epitopos['Coils'] / (avaliacao_epitopos['Coils'] + avaliacao_epitopos['Helices'] + avaliacao_epitopos['Strands'])
) * 100

# Ordena os epítopos do mais flexível (maior percentual de coils) para o mais rígido
epitopos_flexiveis = avaliacao_epitopos.sort_values(
    by=['Percent_Coils', 'Coils'], 
    ascending=[False, False]
).reset_index(drop=True)

epitopos_flexiveis

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,HTWTEQYKFQADSPKRL,26,42,HCCCCEEEEEECCHHHH,6,5,6,35.294118
1,HTWTEQYKFQADSPKRLS,26,43,HCCCCEEEEEECCHHHHH,6,6,6,33.333333
2,HTWTEQYKFQADSPKRLSA,26,44,HCCCCEEEEEECCHHHHHH,6,7,6,31.578947
3,HTWTEQYKFQADSPKRLSAA,26,45,HCCCCEEEEEECCHHHHHHH,6,8,6,30.000000
4,QYKFQADSPKRLS,31,43,EEEEEECCHHHHH,2,5,6,15.384615
5,QYKFQADSPKRLSA,31,44,EEEEEECCHHHHHH,2,6,6,14.285714
6,QYKFQADSPKRLSAA,31,45,EEEEEECCHHHHHHH,2,7,6,13.333333
7,TRLENIMWKQISN,61,73,CHHHHHHHHHHHH,1,12,0,7.692308


#### Filtrar epítopos que tenham pela % de sua estrutura composta por Coils

In [153]:
valor_corte_flexibilidade = parametros.corte_flexibilidade
print(f"Valor de corte de flexibilidade (Percent_Coils): {valor_corte_flexibilidade}%")

Valor de corte de flexibilidade (Percent_Coils): 35.0%


In [154]:
corte_flexibilidade = valor_corte_flexibilidade
epitopos_filtrados_flex = epitopos_flexiveis[epitopos_flexiveis['Percent_Coils'] >= corte_flexibilidade]

epitopos_filtrados_flex

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,HTWTEQYKFQADSPKRL,26,42,HCCCCEEEEEECCHHHH,6,5,6,35.294118


### Merge de dataframes

In [155]:
# Seleciona apenas as colunas exclusivas de conservação (excluindo 'start' e 'end' repetidos, se houver)
colunas_para_pegar = [c for c in conservacy_result_filtered.columns if c not in ['start', 'end', 'Epitope sequence']]
colunas_para_pegar.append('Epitope sequence') # Garante que a chave está incluída

# Realiza o merge limpo
df_consolidado = pd.merge(
    epitopos_filtrados_flex,
    conservacy_result_filtered[colunas_para_pegar],
    left_on='Epitope',
    right_on='Epitope sequence',
    how='inner'
)

# Remove se sobrou a coluna 'Epitope sequence' duplicada:
if 'Epitope sequence' in df_consolidado.columns and 'Epitope' in df_consolidado.columns:
    df_consolidado = df_consolidado.drop(columns=['Epitope sequence'])

df_consolidado

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils,Epitope name,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,alelos,percent_match
0,HTWTEQYKFQADSPKRL,26,42,HCCCCEEEEEECCHHHH,6,5,6,35.294118,NP 29,17,99.70% (1012/1015),94.12%,100.00%,2,0.825,"HLA-DRB3*01:01, HLA-DRB5*01:01",99.7


In [156]:
epitopos_filtrados_flex.to_csv("epitopos_validados_flexibilidade_tcell2.csv", index=False)